In [1]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

## 1. Load Dataset

In [2]:
PATH = "raw_wholesale_customers.csv"
df = pd.read_csv(PATH)
print(df.head())

   Channel  Region  Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0        2       3  12669  9656     7561     214              2674        1338
1        2       3   7057  9810     9568    1762              3293        1776
2        2       3   6353  8808     7684    2405              3516        7844
3        1       3  13265  1196     4221    6404               507        1788
4        2       3  22615  5410     7198    3915              1777        5185


## 2. Select Features + IQR Cap

In [4]:
FEATURES = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
X = df[FEATURES].copy()



def iqr_capping(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    higher = q3 + 1.5 * iqr
    return lower, higher

low_fresh,  high_fresh = iqr_capping(X["Fresh"])
low_milk,    high_milk = iqr_capping(X["Milk"])
low_grocery, high_grocery = iqr_capping(X["Grocery"])
low_frozen,  high_frozen = iqr_capping(X["Frozen"])
low_det_paper,     high_det_paper = iqr_capping(X["Detergents_Paper"])
low_deli,    high_deli = iqr_capping(X["Delicassen"])


X["Fresh"] = X["Fresh"].clip(lower=low_fresh, upper=high_fresh)
X["Milk"] = X["Milk"].clip(lower=low_milk,    upper=high_milk)
X["Grocery"] = X["Grocery"].clip(lower=low_grocery, upper=high_grocery)
X["Frozen"] = X["Frozen"].clip(lower=low_frozen,  upper=high_frozen)
X["Detergents_Paper"] = X["Detergents_Paper"].clip(
    lower=low_det_paper, upper=high_det_paper)
X["Delicassen"] = X["Delicassen"].clip(lower=low_deli, upper=high_deli)

df[FEATURES] = X

print(X.head())


     Fresh    Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0  12669.0  9656.0   7561.0   214.0            2674.0     1338.00
1   7057.0  9810.0   9568.0  1762.0            3293.0     1776.00
2   6353.0  8808.0   7684.0  2405.0            3516.0     3938.25
3  13265.0  1196.0   4221.0  6404.0             507.0     1788.00
4  22615.0  5410.0   7198.0  3915.0            1777.0     3938.25


## 3. Scale Features

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("X scaled", X_scaled)

X scaled [[ 0.12857261  1.05158597  0.04926747 -0.95324427  0.09579175  0.06589216]
 [-0.42162716  1.08673463  0.35386453 -0.30973493  0.30651872  0.47075856]
 [-0.49064723  0.85804007  0.06793486 -0.04243744  0.38243489  2.46943983]
 ...
 [ 0.31112285  2.38267048  2.45460914 -0.86054235  2.39229863  0.55487464]
 [-0.10466425 -0.70014133 -0.7595007  -0.61070442 -0.75732904  0.7933576 ]
 [-0.84025742 -0.76473271 -0.71730938 -1.01518413 -0.65213577 -1.1228252 ]]


# 4 Elbow Method

In [6]:
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init="auto", random_state=42)
    km.fit(X_scaled)
    print(f"k={k} → SSE={km.inertia_:.3f}")

k=1 → SSE=2640.000
k=2 → SSE=1728.189
k=3 → SSE=1363.448
k=4 → SSE=1202.411
k=5 → SSE=1070.147
k=6 → SSE=964.760
k=7 → SSE=921.143
k=8 → SSE=776.632
k=9 → SSE=726.876
k=10 → SSE=707.412


## 5. Train K-Means

In [7]:
K = 3
kmeans = KMeans(n_clusters=K, n_init="auto", random_state=42)
km_labels = kmeans.fit_predict(X_scaled)
df["Cluster"] = km_labels.astype(int)
print(df.head())

   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   
3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   
4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   

   Delicassen  Cluster  
0     1338.00        1  
1     1776.00        0  
2     3938.25        2  
3     1788.00        2  
4     3938.25        2  


## 6. Evaluate K-Means

In [8]:
silh_score = silhouette_score(X_scaled, km_labels)
davbi_score = davies_bouldin_score(X_scaled, km_labels)

# Get K-Means cluster centers in scaled form
scaled_centers = kmeans.cluster_centers_
original_centers = scaler.inverse_transform(scaled_centers)
cluster_centers_df = pd.DataFrame(original_centers,columns=X.columns)
cluster_centers_df.index.name = "Cluster"
print(cluster_centers_df.head())

                Fresh          Milk       Grocery       Frozen  \
Cluster                                                          
0         5869.227723  10142.250000  16653.318069  1355.633663   
1         9418.366466   2685.469880   3583.975904  2052.328313   
2        22881.830556   5870.347222   6773.443056  5057.433333   

         Detergents_Paper   Delicassen  
Cluster                                 
0             7022.053218  1482.601485  
1              942.489960   750.714859  
2             1209.376389  2452.044444  


## 7. Research and train a second Cluster algorithm

In [9]:
agglomerative = AgglomerativeClustering(n_clusters=K)
agg_labels = agglomerative.fit_predict(X_scaled)
df["Agg_cluster"] = agg_labels
print("==========HEAD WITH AGG_CLUSTER==========")
print(df.head())

==========HEAD WITH AGG_CLUSTER==========
   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   
3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   
4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   

   Delicassen  Cluster  Agg_cluster  
0     1338.00        1            2  
1     1776.00        0            2  
2     3938.25        2            0  
3     1788.00        2            0  
4     3938.25        2            0  


## 8. Compare Methods

In [12]:
km_score = silhouette_score(X_scaled, km_labels)
agg_score = silhouette_score(X_scaled, agg_labels)
print(f"K-Means Silhouette Score: {km_score:.3f}")
print(f"Agglomerative Clustering Silhouette Score: {agg_score:.3f}")

K-Means Silhouette Score: 0.340
Agglomerative Clustering Silhouette Score: 0.284


## 9. Sanity Check

In [13]:
sample_idx = [0, 1, 2]
sanity = df.loc[sample_idx, ["Channel", "Region"] + FEATURES + ["Cluster"]]
print(sanity)

   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   

   Delicassen  Cluster  
0     1338.00        1  
1     1776.00        0  
2     3938.25        2  


## 10. Final Output

In [14]:
df.to_csv("segmented_wholesale_customers.csv", index=False)
print(df.head())

   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   
3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   
4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   

   Delicassen  Cluster  Agg_cluster  
0     1338.00        1            2  
1     1776.00        0            2  
2     3938.25        2            0  
3     1788.00        2            0  
4     3938.25        2            0  
